In [10]:
import pandas as pd
import numpy as np
import os
import time
from sklearn.preprocessing import LabelEncoder
import json
import warnings

# Menonaktifkan beberapa jenis peringatan
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
pd.options.mode.chained_assignment = None

# --- ------------------------------------------------------------------ --- #
# ---               KONFIGURASI UTAMA (SESUAIKAN DI SINI)                --- #
# --- ------------------------------------------------------------------ --- #

# 1. Path ke File Data Mentah Anda (bisa .xlsx atau .csv)
RAW_DATA_FILE = 'D:/Kuliah-temp/Teknofest/program/dataset/raw-scanner/5G_DL_processed_full_imputed_v2.xlsx'  # <--- GANTI INI!

# 2. Path untuk File Excel/CSV Output yang Sudah Diagregasi DAN Di-encode
FINAL_OUTPUT_FILE = 'D:/Kuliah-temp/Teknofest/program/dataset/encoded/full.xlsx' # <--- GANTI INI

# 3. Path untuk File JSON Penyimpanan Mapping Encoding
MAPPING_OUTPUT_FILE = 'D:/Kuliah-temp/Teknofest/program/dataset/encoded/full.json' # <--- GANTI INI

# 4. Durasi Jendela Waktu untuk Agregasi
TIME_WINDOW_DURATION = '1s'  # <--- SESUAIKAN INI (misal '2S')

# 5. Kolom yang digunakan untuk mengelompokkan SEBELUM jendela waktu (jika perlu)
#    Jika Lat/Lon sama persis dalam jendela waktu di lokasi yg sama, gunakan ini.
#    Jika trajektori kontinu, bisa dikosongkan.
GROUPBY_COLS_BEFORE_TIMEWINDOW = ['Longitude', 'Latitude'] # Contoh, sesuaikan

# 6. Nilai String Default untuk Kolom Kategorikal Jika Semua Nilai NaN dalam Jendela
#    Ini juga akan menjadi string yang di-encode oleh LabelEncoder
DEFAULT_STRING_FOR_ALL_NAN_CATEGORY = "AGG_MISSING"

# 7. Definisi Agregasi per Kolom
#    Metode agregasi akan diterapkan PADA DATA MENTAH. Encoding dilakukan SETELAHNYA.
COLUMNS_TO_AGGREGATE = { # <--- SESUAIKAN DAFTAR DAN METODE INI
    # 'Longitude': 'first',# Atau 'median'
    # 'Latitude': 'first', # Atau 'median' jika ada variasi kecil dan ingin rata-rata
    
    # Fitur Kategorikal (akan di-encode nanti) - ambil modus dari nilai valid
    'Technology_Mode': lambda x: x.dropna().mode()[0] if not x.dropna().mode().empty else DEFAULT_STRING_FOR_ALL_NAN_CATEGORY,
    'NR_UE_Modulation_Avg_DL_0': lambda x: x.dropna().mode()[0] if not x.dropna().mode().empty else DEFAULT_STRING_FOR_ALL_NAN_CATEGORY,
    'NR_UE_CCE_AggregationLev_0': lambda x: x.dropna().mode()[0] if not x.dropna().mode().empty else np.nan, # Ini mungkin numerik, jadi NaN jika semua NaN
    'NR_UE_RI_DL_0': lambda x: x.dropna().mode()[0] if not x.dropna().mode().empty else np.nan, # Ini mungkin numerik

    # Parameter Serving Cell Numerik
    'NR_UE_PCI_0': lambda x: x.dropna().mode()[0] if not x.dropna().mode().empty else np.nan, # PCI serving, modus jika ada beberapa
    'NR_UE_RSRP_0': 'median',
    'NR_UE_RSRQ_0': 'median',
    'NR_UE_SINR_0': 'median',
    'NR_UE_Timing_Advance': 'median',
    'NR_UE_Pathloss_DL_0': 'median',

    # Parameter Neighbor Cells (ambil modus PCI, median pengukuran)
    'NR_UE_Nbr_PCI_0': lambda x: x.dropna().mode()[0] if not x.dropna().mode().empty else np.nan,
    'NR_UE_Nbr_RSRP_0': 'median', 'NR_UE_Nbr_RSRQ_0': 'median',
    'NR_UE_Nbr_PCI_1': lambda x: x.dropna().mode()[0] if not x.dropna().mode().empty else np.nan,
    'NR_UE_Nbr_RSRP_1': 'median', 'NR_UE_Nbr_RSRQ_1': 'median',
    # ... dan seterusnya untuk semua neighbor PCI, RSRP, RSRQ ...
    'NR_UE_Nbr_PCI_4': lambda x: x.dropna().mode()[0] if not x.dropna().mode().empty else np.nan,
    'NR_UE_Nbr_RSRP_4': 'median', 'NR_UE_Nbr_RSRQ_4': 'median',

    # Parameter Performa & Error
    'NR_UE_Throughput_PDCP_DL': 'mean',
    'App_Throughput_DL': 'mean',
    'NR_UE_NACK_Rate_DL_0': 'mean',
    'NR_UE_BLER_DL_0': 'mean',
    'NR_UE_Power_Tx_PUSCH_0': 'median',
    'NR_UE_NACK_Rate_UL_0': 'mean',

    # Parameter RACH & RRC Counters
    'NR_UE_RACH_Attempt': 'sum',
    'NR_UE_RACH_OK': 'sum',
    # ... tambahkan semua kolom lain yang ingin diagregasi ...
}

# 8. Kolom (HASIL AGREGASI) yang Akan Dilakukan Label Encoding
#    Ini adalah nama kolom SETELAH agregasi yang masih berupa string/object.
COLUMNS_TO_LABEL_ENCODE_POST_AGG = [ # <--- SESUAIKAN DAFTAR INI
    'Technology_Mode',
    'NR_UE_Modulation_Avg_DL_0',
    'NR_UE_CCE_AggregationLev_0',
    'NR_UE_RI_DL_0'
    # Tambahkan kolom lain hasil agregasi yang perlu di-encode
    # Contoh: 'NR_UE_RRCReEst_EndResult' jika Anda mengagregasinya
]
# Nilai placeholder untuk NaN SEBELUM encoding pada tahap Label Encoding (jika ada NaN setelah agregasi)
NAN_PLACEHOLDER_FOR_LABEL_ENCODING = "FINAL_MISSING"

# --- ------------------------------------------------------------------ --- #
# ---                      AKHIR KONFIGURASI UTAMA                       --- #
# --- ------------------------------------------------------------------ --- #

def aggregate_data_by_time_window(df_raw, time_window, agg_config, group_cols):
    print(f"Memulai agregasi data dengan jendela waktu: {time_window}...")
    start_time = time.time()
    if 'Time' not in df_raw.columns: print("FATAL: Kolom 'Time' hilang."); return None
    try: df_raw['Time'] = pd.to_datetime(df_raw['Time'])
    except Exception as e: print(f"FATAL: Gagal konversi 'Time': {e}"); return None

    valid_agg_config = {k: v for k, v in agg_config.items() if k in df_raw.columns}
    if not valid_agg_config: print("FATAL: Tidak ada kolom valid untuk diagregasi."); return None
    print(f"  Kolom yang akan diagregasi: {list(valid_agg_config.keys())}")

    # Pra-konversi tipe data untuk kolom yang akan diagregasi dengan fungsi numerik
    for col, agg_method in valid_agg_config.items():
        if isinstance(agg_method, str) and agg_method in ['mean', 'median', 'sum', 'min', 'max', 'std', 'var']:
            if col in df_raw.columns:
                df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

    sort_by_cols = group_cols + ['Time'] if group_cols else ['Time']
    df_raw_sorted = df_raw.sort_values(by=sort_by_cols)
    
    print(f"  Mengelompokkan berdasarkan: {group_cols} dan jendela waktu pada 'Time'...")
    time_grouper = pd.Grouper(key='Time', freq=time_window)
    all_group_by_keys = group_cols + [time_grouper] if group_cols else [time_grouper]

    try:
        df_aggregated = df_raw_sorted.groupby(all_group_by_keys, observed=True).agg(valid_agg_config)
    except Exception as e:
        print(f"FATAL: Error saat agregasi: {e}"); import traceback; traceback.print_exc(); return None
        
    df_aggregated = df_aggregated.reset_index()
    elapsed_time = time.time() - start_time
    print(f"Agregasi data selesai dalam {elapsed_time:.2f}s. Bentuk: {df_aggregated.shape}")
    return df_aggregated

def encode_categorical_columns_sklearn(df_input, columns_to_encode, nan_placeholder):
    df_encoded = df_input.copy()
    fitted_encoders = {}
    explicit_mappings = {}
    print("\nMemulai Label Encoding pada data agregat...")
    for col in columns_to_encode:
        if col not in df_encoded.columns:
            print(f"Peringatan: Kolom '{col}' untuk encoding tidak ditemukan di DataFrame agregat. Dilewati.")
            continue
        print(f"  Encoding kolom: '{col}'")
        df_encoded[col] = df_encoded[col].fillna(nan_placeholder).astype(str) # Pastikan string dan handle NaN
        encoder = LabelEncoder()
        df_encoded[col] = encoder.fit_transform(df_encoded[col])
        fitted_encoders[col] = encoder
        explicit_mappings[col] = {original_class: int_code for int_code, original_class in enumerate(encoder.classes_)}
        print(f"    Kolom '{col}' selesai di-encode. Jumlah kelas unik: {len(encoder.classes_)}")
    return df_encoded, fitted_encoders, explicit_mappings

if __name__ == '__main__':
    print("--- Skrip Agregasi dan Encoding Dimulai ---")

    # 1. Baca data mentah
    print(f"\n[Langkah 1] Membaca data mentah dari: {RAW_DATA_FILE}")
    try:
        if RAW_DATA_FILE.endswith('.xlsx'):
            df_raw_input = pd.read_excel(RAW_DATA_FILE) # Tambahkan sheet_name jika perlu
        elif RAW_DATA_FILE.endswith('.csv'):
            df_raw_input = pd.read_csv(RAW_DATA_FILE, low_memory=False)
        else: print("FATAL: Format file tidak didukung."); exit()
        print(f"  Data mentah berhasil dimuat: {len(df_raw_input)} baris.")
    except FileNotFoundError: print(f"FATAL: File mentah '{RAW_DATA_FILE}' tidak ditemukan."); exit()
    except Exception as e: print(f"FATAL: Error membaca data mentah: {e}"); exit()
    if df_raw_input.empty: print("FATAL: Data mentah kosong."); exit()

    df_raw_input.columns = df_raw_input.columns.str.strip()
    COLUMNS_TO_AGGREGATE = {k.strip(): v for k, v in COLUMNS_TO_AGGREGATE.items()}
    GROUPBY_COLS_BEFORE_TIMEWINDOW = [col.strip() for col in GROUPBY_COLS_BEFORE_TIMEWINDOW]
    COLUMNS_TO_LABEL_ENCODE_POST_AGG = [col.strip() for col in COLUMNS_TO_LABEL_ENCODE_POST_AGG]

    # 2. Lakukan agregasi per jendela waktu
    df_aggregated = aggregate_data_by_time_window(
        df_raw_input,
        TIME_WINDOW_DURATION,
        COLUMNS_TO_AGGREGATE,
        GROUPBY_COLS_BEFORE_TIMEWINDOW
    )

    if df_aggregated is None or df_aggregated.empty:
        print("Agregasi gagal atau menghasilkan DataFrame kosong. Skrip dihentikan.")
        exit()

    # 3. Lakukan Label Encoding pada kolom kategorikal hasil agregasi
    df_final_encoded, final_encoders, final_mappings = encode_categorical_columns_sklearn(
        df_aggregated,
        COLUMNS_TO_LABEL_ENCODE_POST_AGG,
        NAN_PLACEHOLDER_FOR_LABEL_ENCODING
    )
    
    # (Opsional) Mengisi NaN yang mungkin masih ada di kolom numerik setelah agregasi
    # Misalnya, jika semua nilai dalam jendela untuk kolom numerik adalah NaN, hasil agregasi (median/mean) akan NaN.
    print("\n(Opsional) Mengisi NaN yang tersisa di kolom numerik dengan 0...")
    for col in df_final_encoded.columns:
        if df_final_encoded[col].dtype in [np.float64, np.int64, float, int]: # Cek jika numerik
             if df_final_encoded[col].isnull().any():
                print(f"  Mengisi NaN di kolom numerik '{col}' dengan 0.")
                df_final_encoded[col].fillna(0, inplace=True) # Ganti 0 dengan strategi imputasi lain jika perlu

    # 4. Simpan hasil akhir (diagregasi dan di-encode)
    if df_final_encoded is not None and not df_final_encoded.empty:
        print(f"\n[Langkah 4] Menyimpan data akhir ke: {FINAL_OUTPUT_FILE}")
        try:
            output_dir_final = os.path.dirname(FINAL_OUTPUT_FILE)
            if output_dir_final and not os.path.exists(output_dir_final):
                os.makedirs(output_dir_final)
            
            if FINAL_OUTPUT_FILE.endswith('.xlsx'):
                df_final_encoded.to_excel(FINAL_OUTPUT_FILE, index=False, engine='openpyxl')
            elif FINAL_OUTPUT_FILE.endswith('.csv'):
                df_final_encoded.to_csv(FINAL_OUTPUT_FILE, index=False)
            else:
                print("FATAL: Format file output tidak didukung untuk penyimpanan.")
            print(f"  Data akhir berhasil disimpan.")
        except Exception as e:
            print(f"FATAL: Gagal menyimpan data akhir: {e}")
    else:
        print("Proses gagal atau menghasilkan DataFrame kosong, tidak ada yang disimpan.")

    # 5. Simpan mapping encoding
    if final_mappings:
        print(f"\nMenyimpan mapping encoding ke: {MAPPING_OUTPUT_FILE}")
        try:
            output_dir_json = os.path.dirname(MAPPING_OUTPUT_FILE)
            if output_dir_json and not os.path.exists(output_dir_json):
                os.makedirs(output_dir_json)
            with open(MAPPING_OUTPUT_FILE, 'w') as f:
                json.dump(final_mappings, f, indent=4)
            print(f"  Mapping encoding berhasil disimpan.")
        except Exception as e:
            print(f"FATAL: Gagal menyimpan mapping encoding: {e}")
    else:
        print("Tidak ada mapping encoding yang dibuat.")

    print("\n--- Skrip Agregasi dan Encoding Selesai ---")

--- Skrip Agregasi dan Encoding Dimulai ---

[Langkah 1] Membaca data mentah dari: D:/Kuliah-temp/Teknofest/program/dataset/raw-scanner/5G_DL_processed_full_imputed_v2.xlsx
  Data mentah berhasil dimuat: 72480 baris.
Memulai agregasi data dengan jendela waktu: 1s...
  Kolom yang akan diagregasi: ['Technology_Mode', 'NR_UE_Modulation_Avg_DL_0', 'NR_UE_CCE_AggregationLev_0', 'NR_UE_RI_DL_0', 'NR_UE_PCI_0', 'NR_UE_RSRP_0', 'NR_UE_RSRQ_0', 'NR_UE_SINR_0', 'NR_UE_Timing_Advance', 'NR_UE_Pathloss_DL_0', 'NR_UE_Nbr_PCI_0', 'NR_UE_Nbr_RSRP_0', 'NR_UE_Nbr_RSRQ_0', 'NR_UE_Nbr_PCI_1', 'NR_UE_Nbr_RSRP_1', 'NR_UE_Nbr_RSRQ_1', 'NR_UE_Nbr_PCI_4', 'NR_UE_Nbr_RSRP_4', 'NR_UE_Nbr_RSRQ_4', 'NR_UE_Throughput_PDCP_DL', 'App_Throughput_DL', 'NR_UE_NACK_Rate_DL_0', 'NR_UE_BLER_DL_0', 'NR_UE_Power_Tx_PUSCH_0', 'NR_UE_NACK_Rate_UL_0', 'NR_UE_RACH_Attempt', 'NR_UE_RACH_OK']
  Mengelompokkan berdasarkan: ['Longitude', 'Latitude'] dan jendela waktu pada 'Time'...
Agregasi data selesai dalam 2.20s. Bentuk: (2131, 

In [ ]:
import pandas as pd
import numpy as np
import os
import time

# --- ------------------------------------------------------------------ --- #
# ---               KONFIGURASI UTAMA (SESUAIKAN DI SINI)                --- #
# --- ------------------------------------------------------------------ --- #

# 1. Path ke File Data Mentah Anda (bisa .xlsx atau .csv)
#    Ini adalah file yang ingin Anda agregasi per Lat/Lon unik.
#    Bisa jadi file mentah Anda, atau file yang sudah melalui tahap encoding/pembersihan awal.
RAW_DATA_FILE = "D:/Kuliah-temp/Teknofest/program/dataset/encoded/encoded_5G_UL_fill.xlsx" # Ganti dengan path Anda

# 2. Path untuk File Excel Output yang Sudah Diagregasi per Lokasi
AGGREGATED_PER_LOCATION_OUTPUT_FILE = 'aggregated_data_per_location_v1.xlsx' # Ganti nama output

# 3. Kolom yang Digunakan untuk Mendefinisikan Lokasi Unik (Kunci GroupBy)
#    PENTING: Pastikan nilai di kolom ini cukup presisi untuk membedakan lokasi yang benar-benar berbeda.
#             Jika Lat/Lon Anda memiliki noise GPS, Anda mungkin perlu membulatkannya ke sejumlah desimal tertentu
#             SEBELUM melakukan groupby ini (misalnya, buat kolom baru Lat_rounded, Lon_rounded).
LOCATION_GROUPBY_COLUMNS = ['Longitude', 'Latitude'] # <--- INI KUNCI PERUBAHANNYA

# 4. Definisi Agregasi per Kolom (Metode agregasi akan diterapkan pada semua sampel di Lat/Lon yang sama)
COLUMNS_TO_AGGREGATE_PER_LOCATION = { # <--- SESUAIKAN DAFTAR DAN METODE INI
    # Kolom identifikasi 'Latitude' dan 'Longitude' akan menjadi indeks grup,
    # jadi kita tidak perlu mengagregasinya lagi di sini, KECUALI jika Anda melakukan pembulatan
    # dan ingin menyimpan nilai Lat/Lon asli pertama atau rata-ratanya.
    # Jika LOCATION_GROUPBY_COLUMNS sudah presisi, kita bisa hilangkan dari sini.
    # Mari kita asumsikan kita ingin nilai 'first' dari Lat/Lon asli (jika ada pembulatan sebelumnya)
    # atau jika tidak ada pembulatan, ini akan jadi nilai itu sendiri.
    # 'Longitude': 'first', # Akan menjadi bagian dari indeks jika tidak ada pembulatan
    # 'Latitude': 'first',  # Akan menjadi bagian dari indeks jika tidak ada pembulatan

    # Contoh jika Anda ingin menghitung jumlah sampel asli per lokasi:
    # 'Time': 'count', # Akan memberi nama kolom 'Time' dengan jumlah sampel

    # Parameter Serving Cell
    'Technology_Mode': lambda x: x.mode()[0] if not x.mode().empty else '0',
    'NR_UE_PCI_0': lambda x: x.mode()[0] if not x.mode().empty else np.nan, # Modus untuk PCI serving
    'NR_UE_RSRP_0': 'median',
    'NR_UE_RSRQ_0': 'median',
    'NR_UE_SINR_0': 'median',
    'NR_UE_Timing_Advance': 'median',
    'NR_UE_Pathloss_DL_0': 'median',
    'NR_UE_CCE_AggregationLev_0': lambda x: x.mode()[0] if not x.mode().empty else np.nan, # Modus
    'NR_UE_Modulation_Avg_DL_0': lambda x: x.mode()[0] if not x.mode().empty else np.nan, # Modus
    'NR_UE_RI_DL_0': lambda x: x.mode()[0] if not x.mode().empty else np.nan, # Modus

    # Parameter Neighbor Cells
    # Untuk neighbor, mengambil modus PCI dan median RSRP/RSRQ dari semua sampel di lokasi itu
    # bisa jadi representasi yang baik.
    'NR_UE_Nbr_PCI_0': lambda x: x.mode()[0] if not x.mode().empty else np.nan,
    'NR_UE_Nbr_RSRP_0': 'median', 'NR_UE_Nbr_RSRQ_0': 'median',
    'NR_UE_Nbr_PCI_1': lambda x: x.mode()[0] if not x.mode().empty else np.nan,
    'NR_UE_Nbr_RSRP_1': 'median', 'NR_UE_Nbr_RSRQ_1': 'median',
    # ... dan seterusnya untuk semua neighbor ...
    'NR_UE_Nbr_PCI_4': lambda x: x.mode()[0] if not x.mode().empty else np.nan,
    'NR_UE_Nbr_RSRP_4': 'median', 'NR_UE_Nbr_RSRQ_4': 'median',

    # Parameter Performa & Error
    'NR_UE_Throughput_PDCP_DL': 'mean',
    'App_Throughput_DL': 'mean',
    'NR_UE_NACK_Rate_DL_0': 'mean',
    'NR_UE_Ack_As_Nack_DL_0': 'mean',
    'NR_UE_BLER_DL_0': 'mean',
    'NR_UE_Power_Tx_PUSCH_0': 'median',
    'NR_UE_Power_Tx_PRACH_0': 'median',
    'NR_UE_NACK_Rate_UL_0': 'mean',

    # Parameter RACH & RRC
    'NR_UE_RACH_Attempt': 'sum', # Total percobaan RACH di lokasi itu
    'NR_UE_RACH_OK': 'sum',
    'NR_UE_RACH_Fail': 'sum',
    'NR_UE_RRCReEstAttempt': 'sum',
    'NR_UE_RRCReEstFail': 'sum',
    # 'NR_UE_RRCReEst_EndResult': lambda x: x.mode()[0] if not x.mode().empty else 'Unknown',
    'NR_UE_RRCConnectionDrop': 'sum',
    'NR_UE_RRCHOAttempt': 'sum',
    'NR_UE_RRCHOOK': 'sum',
}

# --- ------------------------------------------------------------------ --- #
# ---                      AKHIR KONFIGURASI UTAMA                       --- #
# --- ------------------------------------------------------------------ --- #

def aggregate_data_per_location(df_raw, location_group_cols, agg_config):
    """
    Mengagregasi DataFrame berdasarkan kolom lokasi unik.
    """
    print(f"Memulai agregasi data per lokasi unik menggunakan kolom: {location_group_cols}...")
    start_time = time.time()

    # Pastikan kolom lokasi untuk groupby ada
    missing_group_cols = [col for col in location_group_cols if col not in df_raw.columns]
    if missing_group_cols:
        print(f"FATAL: Kolom untuk groupby lokasi tidak ditemukan: {missing_group_cols}")
        return None

    # Pra-pemrosesan tipe data untuk kolom yang akan diagregasi numerik
    # (Mirip dengan fungsi agregasi waktu, untuk memastikan fungsi numerik tidak error)
    print("  Melakukan pra-konversi tipe data untuk kolom numerik...")
    df_to_agg = df_raw.copy() # Bekerja pada salinan
    for col, agg_method in agg_config.items():
        if col in df_to_agg.columns:
            if isinstance(agg_method, str) and agg_method in ['mean', 'median', 'sum', 'min', 'max', 'std', 'var']:
                df_to_agg[col] = pd.to_numeric(df_to_agg[col], errors='coerce')
                if df_to_agg[col].isnull().all() and not df_to_agg[col].empty:
                    print(f"    Peringatan: Semua nilai di kolom '{col}' menjadi NaN setelah konversi ke numerik.")
            # Untuk modus dan 'first'/'last'/'count'/'nunique', tipe data asli biasanya tidak masalah,
            # atau lambda function sudah menanganinya.

    # Hapus kolom dari agg_config yang tidak ada di df_to_agg setelah pengecekan
    valid_agg_config = {k: v for k, v in agg_config.items() if k in df_to_agg.columns}
    if not valid_agg_config:
        print("FATAL: Tidak ada kolom valid yang tersisa untuk diagregasi.")
        return None
    print(f"  Kolom yang akan diagregasi: {list(valid_agg_config.keys())}")

    print(f"  Mengelompokkan berdasarkan: {location_group_cols}...")
    try:
        # Lakukan groupby pada kolom lokasi dan agregasi
        df_aggregated = df_to_agg.groupby(location_group_cols, observed=True).agg(valid_agg_config)
    except Exception as e:
        print(f"FATAL: Error saat melakukan agregasi per lokasi: {e}")
        import traceback
        traceback.print_exc()
        return None
        
    # Reset index untuk mengubah kunci grup (Lat, Lon) menjadi kolom biasa
    df_aggregated = df_aggregated.reset_index()
    
    elapsed_time = time.time() - start_time
    print(f"Agregasi data per lokasi selesai dalam {elapsed_time:.2f} detik.")
    print(f"Bentuk DataFrame setelah agregasi per lokasi: {df_aggregated.shape}")
    return df_aggregated

if __name__ == '__main__':
    print("--- Skrip Agregasi Data per Lokasi Unik Dimulai ---")

    # Baca data mentah
    print(f"Membaca data mentah dari: {RAW_DATA_FILE}")
    try:
        if RAW_DATA_FILE.endswith('.xlsx'):
            # Jika file Excel Anda memiliki nama sheet spesifik:
            df_raw_input = pd.read_excel(RAW_DATA_FILE, sheet_name="Sheet1") # Ganti "Sheet1" jika perlu
        elif RAW_DATA_FILE.endswith('.csv'):
            df_raw_input = pd.read_csv(RAW_DATA_FILE, low_memory=False)
        else:
            print("FATAL: Format file tidak didukung (harus .xlsx atau .csv)."); exit()
        print(f"Data mentah berhasil dimuat: {len(df_raw_input)} baris.")
    except FileNotFoundError: print(f"FATAL: File mentah tidak ditemukan: {RAW_DATA_FILE}"); exit()
    except Exception as e: print(f"FATAL: Error saat membaca data mentah: {e}"); exit()

    if df_raw_input.empty: print("FATAL: Data mentah yang dimuat kosong."); exit()

    # PENTING: Cek dan bersihkan nama kolom dari spasi ekstra
    df_raw_input.columns = df_raw_input.columns.str.strip()
    # Perbarui juga COLUMNS_TO_AGGREGATE_PER_LOCATION jika nama kuncinya ada spasi
    COLUMNS_TO_AGGREGATE_PER_LOCATION = {k.strip(): v for k, v in COLUMNS_TO_AGGREGATE_PER_LOCATION.items()}
    LOCATION_GROUPBY_COLUMNS = [col.strip() for col in LOCATION_GROUPBY_COLUMNS]


    # Opsional: Jika Lat/Lon Anda memiliki noise dan ingin membulatkannya untuk groupby
    # PRECISION = 6 # Jumlah angka di belakang koma untuk pembulatan
    # df_raw_input['Lat_rounded'] = df_raw_input['Latitude'].round(PRECISION)
    # df_raw_input['Lon_rounded'] = df_raw_input['Longitude'].round(PRECISION)
    # LOCATION_GROUPBY_COLUMNS_FOR_SCRIPT = ['Lat_rounded', 'Lon_rounded']
    # # Jika melakukan ini, Anda mungkin ingin menambahkan 'Latitude': 'first', 'Longitude': 'first'
    # # ke COLUMNS_TO_AGGREGATE_PER_LOCATION untuk menyimpan nilai Lat/Lon asli pertama.
    LOCATION_GROUPBY_COLUMNS_FOR_SCRIPT = LOCATION_GROUPBY_COLUMNS # Default


    # Lakukan agregasi per lokasi
    df_aggregated_output = aggregate_data_per_location(
        df_raw_input,
        LOCATION_GROUPBY_COLUMNS_FOR_SCRIPT, # Kolom yang digunakan untuk groupby
        COLUMNS_TO_AGGREGATE_PER_LOCATION
    )

    # Simpan hasil agregasi ke Excel
    if df_aggregated_output is not None and not df_aggregated_output.empty:
        print(f"Menyimpan data yang sudah diagregasi per lokasi ke: {AGGREGATED_PER_LOCATION_OUTPUT_FILE}")
        try:
            output_dir = os.path.dirname(AGGREGATED_PER_LOCATION_OUTPUT_FILE)
            if output_dir and not os.path.exists(output_dir):
                os.makedirs(output_dir)
            df_aggregated_output.to_excel(AGGREGATED_PER_LOCATION_OUTPUT_FILE, index=False, engine='openpyxl')
            print("Data yang sudah diagregasi per lokasi berhasil disimpan.")
        except Exception as e:
            print(f"FATAL: Gagal menyimpan data yang sudah diagregasi per lokasi: {e}")
            print("Pastikan Anda memiliki library 'openpyxl' terinstal: pip install openpyxl")
    else:
        print("Agregasi data per lokasi gagal atau menghasilkan DataFrame kosong, tidak ada yang disimpan.")
        
    print("--- Skrip Agregasi Data per Lokasi Unik Selesai ---")

--- Skrip Agregasi Data per Lokasi Unik Dimulai ---
Membaca data mentah dari: D:/Kuliah-temp/Teknofest/program/dataset/encoded/encoded_5G_UL_fill.xlsx
Data mentah berhasil dimuat: 72480 baris.
Memulai agregasi data per lokasi unik menggunakan kolom: ['Longitude', 'Latitude']...
  Melakukan pra-konversi tipe data untuk kolom numerik...
  Kolom yang akan diagregasi: ['Technology_Mode', 'NR_UE_PCI_0', 'NR_UE_RSRP_0', 'NR_UE_RSRQ_0', 'NR_UE_SINR_0', 'NR_UE_Timing_Advance', 'NR_UE_Pathloss_DL_0', 'NR_UE_CCE_AggregationLev_0', 'NR_UE_Modulation_Avg_DL_0', 'NR_UE_RI_DL_0', 'NR_UE_Nbr_PCI_0', 'NR_UE_Nbr_RSRP_0', 'NR_UE_Nbr_RSRQ_0', 'NR_UE_Nbr_PCI_1', 'NR_UE_Nbr_RSRP_1', 'NR_UE_Nbr_RSRQ_1', 'NR_UE_Nbr_PCI_4', 'NR_UE_Nbr_RSRP_4', 'NR_UE_Nbr_RSRQ_4', 'NR_UE_Throughput_PDCP_DL', 'App_Throughput_DL', 'NR_UE_NACK_Rate_DL_0', 'NR_UE_Ack_As_Nack_DL_0', 'NR_UE_BLER_DL_0', 'NR_UE_Power_Tx_PUSCH_0', 'NR_UE_Power_Tx_PRACH_0', 'NR_UE_NACK_Rate_UL_0', 'NR_UE_RACH_Attempt', 'NR_UE_RACH_OK', 'NR_UE_RACH_Fail'

In [6]:
import pandas as pd
import numpy as np
import os
import time

# --- ------------------------------------------------------------------ --- #
# ---               KONFIGURASI UTAMA (SESUAIKAN DI SINI)                --- #
# --- ------------------------------------------------------------------ --- #

# 1. Path ke File Data Mentah Anda (bisa .xlsx atau .csv)
RAW_DATA_FILE = "D:/Kuliah-temp/Teknofest/program/dataset/encoded/encoded_5G_UL_fill.xlsx"
DL_DATASET_PATH = "D:/Kuliah-temp/Teknofest/program/dataset/raw-scanner/5G_DL.xlsx"


# 2. Path untuk File Excel Output yang Sudah Diagregasi
AGGREGATED_OUTPUT_FILE = 'aggregated_data_per_time_window_fill2.xlsx' # <--- GANTI INI

# 3. Durasi Jendela Waktu untuk Agregasi
#    Format string Pandas, contoh: '1S' (1 detik), '2S' (2 detik), '500ms' (500 milidetik)
TIME_WINDOW_DURATION = '1s'  # <--- SESUAIKAN INI

# 4. Definisi Agregasi per Kolom
#    Format: {'NamaKolomDiFileMentah': 'metode_agregasi'}
#    Metode agregasi yang umum: 'mean', 'median', 'sum', 'min', 'max', 'first', 'last', 'count', 'nunique',
#    'std', 'var', atau fungsi lambda kustom (misal, lambda x: x.mode()[0] if not x.mode().empty else np.nan)
#
#    PENTING: Kolom 'Latitude' dan 'Longitude' diasumsikan SAMA PERSIS dalam satu jendela waktu
#             di lokasi yang sama. Jika ada variasi kecil, 'first' atau 'mean' bisa jadi pilihan.
#             Kolom 'Time' akan menjadi awal dari setiap jendela waktu setelah agregasi.

COLUMNS_TO_AGGREGATE = { # <--- SESUAIKAN DAFTAR DAN METODE INI
    # Kolom identifikasi (biasanya 'first' atau 'mean' jika ada variasi kecil)
    'Longitude': 'median',# Asumsi Lon sama dalam jendela di lokasi yg sama
    'Latitude': 'median', # Asumsi Lat sama dalam jendela di lokasi yg sama
    # 'Technology_Mode': lambda x: x.mode()[0] if not x.mode().empty else '0', # Modus untuk kategorikal

    # Parameter Serving Cell
    'NR_UE_PCI_0': 'first', # Atau modus jika bisa berubah dalam 2 detik
    'NR_UE_RSRP_0': 'median',
    'NR_UE_RSRQ_0': 'median',
    'NR_UE_SINR_0': 'median',
    'NR_UE_Timing_Advance': 'median',
    'NR_UE_Pathloss_DL_0': 'median',
    'NR_UE_CCE_AggregationLev_0':'last' ,#lambda x: x.mode()[0] if not x.mode().empty else '0' , # atau modus
    'NR_UE_Modulation_Avg_DL_0': 'first', #lambda x: x.mode()[0] if not x.mode().empty else '0', # Modus
    'NR_UE_RI_DL_0': 'median', # atau modus

    # Parameter Neighbor Cells (kita ambil 'first' untuk PCI dan 'median' untuk pengukuran)
    # Ini berarti kita mengambil PCI neighbor pertama yang dilaporkan dalam jendela itu
    # dan median dari semua pengukuran RSRP/RSRQ untuk slot neighbor tersebut.
    # Alternatif: Anda bisa coba 'last' atau bahkan fungsi kustom yang lebih kompleks.
    'NR_UE_Nbr_PCI_0': 'first', 'NR_UE_Nbr_RSRP_0': 'median', 'NR_UE_Nbr_RSRQ_0': 'median',
    'NR_UE_Nbr_PCI_1': 'first', 'NR_UE_Nbr_RSRP_1': 'median', 'NR_UE_Nbr_RSRQ_1': 'median',
    'NR_UE_Nbr_PCI_2': 'first', 'NR_UE_Nbr_RSRP_2': 'median', 'NR_UE_Nbr_RSRQ_2': 'median',
    'NR_UE_Nbr_PCI_3': 'first', 'NR_UE_Nbr_RSRP_3': 'median', 'NR_UE_Nbr_RSRQ_3': 'median',
    'NR_UE_Nbr_PCI_4': 'first', 'NR_UE_Nbr_RSRP_4': 'median', 'NR_UE_Nbr_RSRQ_4': 'median',

    # Parameter Performa & Error
    'NR_UE_Throughput_PDCP_DL': 'mean', # atau 'sum' jika ingin total throughput
    'App_Throughput_DL': 'mean',       # atau 'sum'
    'NR_UE_NACK_Rate_DL_0': 'mean',
    'NR_UE_Ack_As_Nack_DL_0': 'mean', # atau 'sum'
    'NR_UE_BLER_DL_0': 'mean',
    'NR_UE_Power_Tx_PUSCH_0': 'median',
    'NR_UE_Power_Tx_PRACH_0': 'median', # atau 'max'
    'NR_UE_NACK_Rate_UL_0': 'mean',

    # Parameter RACH & RRC (biasanya 'sum' untuk counter, atau 'first'/'last'/'nunique' untuk status)
    'NR_UE_RACH_Attempt': 'sum',
    'NR_UE_RACH_OK': 'sum',
    'NR_UE_RACH_Fail': 'sum',
    'NR_UE_RRCReEstAttempt': 'sum',
    'NR_UE_RRCReEstFail': 'sum',
    # 'NR_UE_RRCReEst_EndResult': lambda x: x.mode()[0] if not x.mode().empty else 'Unknown',
    'NR_UE_RRCConnectionDrop': 'sum',
    'NR_UE_RRCHOAttempt': 'sum',
    'NR_UE_RRCHOOK': 'sum',

    # Kolom lain yang mungkin ingin Anda sertakan
    # 'NR_UE_MCS_DL_0': 'median',
    # 'NR_UE_RB_Num_DL_0': 'median',
    # ... pastikan semua kolom yang Anda inginkan ada di sini ...
}

# 5. Kolom yang digunakan untuk mengelompokkan SEBELUM jendela waktu
#    Biasanya 'Latitude' dan 'Longitude' jika Anda ingin agregasi per lokasi unik
#    Jika data Anda adalah satu trajektori kontinu, Anda mungkin tidak perlu ini
#    dan hanya menggunakan pd.Grouper pada 'Time'.
#    Jika Lat/Lon sama persis untuk 68 titik, kita bisa group by ini.
GROUPBY_COLS_BEFORE_TIMEWINDOW = [] # <--- SESUAIKAN INI

# --- ------------------------------------------------------------------ --- #
# ---                      AKHIR KONFIGURASI UTAMA                       --- #
# --- ------------------------------------------------------------------ --- #

def aggregate_data_by_time_window(df_raw, time_window, agg_config, group_cols):
    """
    Mengagregasi DataFrame berdasarkan jendela waktu.
    """
    print(f"Memulai agregasi data dengan jendela waktu: {time_window}...")
    start_time = time.time()

    # Pastikan kolom 'Time' adalah datetime
    if 'Time' not in df_raw.columns:
        print("FATAL: Kolom 'Time' tidak ditemukan di dataset.")
        return None
    try:
        df_raw['Time'] = pd.to_datetime(df_raw['Time'])
    except Exception as e:
        print(f"FATAL: Gagal mengonversi kolom 'Time' menjadi datetime: {e}")
        return None

    # Pastikan semua kolom yang akan diagregasi ada di DataFrame
    for col in agg_config.keys():
        if col not in df_raw.columns:
            print(f"Peringatan: Kolom '{col}' untuk agregasi tidak ditemukan di DataFrame mentah. Akan diabaikan.")
            # Hapus dari config agar tidak error
            # agg_config.pop(col) # Sebaiknya jangan modifikasi dict saat iterasi, buat salinan

    valid_agg_config = {k: v for k, v in agg_config.items() if k in df_raw.columns}
    if not valid_agg_config:
        print("FATAL: Tidak ada kolom valid yang tersisa untuk diagregasi.")
        return None
        
    print(f"  Kolom yang akan diagregasi: {list(valid_agg_config.keys())}")

    # Mengurutkan data (penting untuk beberapa metode agregasi seperti 'first', 'last')
    # Jika group_cols kosong, urutkan hanya berdasarkan Waktu
    sort_by_cols = group_cols + ['Time'] if group_cols else ['Time']
    df_raw_sorted = df_raw.sort_values(by=sort_by_cols)
    
    print(f"  Mengelompokkan berdasarkan: {group_cols} dan jendela waktu pada 'Time'...")
    
    # Buat objek Grouper
    time_grouper = pd.Grouper(key='Time', freq=time_window)
    
    # Gabungkan kolom groupby dengan time_grouper
    all_group_by_keys = group_cols + [time_grouper] if group_cols else [time_grouper]

    try:
        df_aggregated = df_raw_sorted.groupby(all_group_by_keys, observed=True).agg(valid_agg_config)
        # 'observed=True' ditambahkan untuk perilaku groupby yang lebih modern pada kolom kategorikal/datetime
    except Exception as e:
        print(f"FATAL: Error saat melakukan agregasi: {e}")
        import traceback
        traceback.print_exc()
        return None
        
    # Reset index untuk mengubah kunci grup (Lat, Lon, Time_window_start) menjadi kolom biasa
    df_aggregated = df_aggregated.reset_index()
    
    # Jika group_cols kosong, kolom 'Time' (awal jendela) sudah ada dari reset_index.
    # Jika group_cols ada, 'Time' juga akan menjadi nama kolom.

    elapsed_time = time.time() - start_time
    print(f"Agregasi data selesai dalam {elapsed_time:.2f} detik.")
    print(f"Bentuk DataFrame setelah agregasi: {df_aggregated.shape}")
    return df_aggregated

if __name__ == '__main__':
    print("--- Skrip Agregasi Data per Jendela Waktu Dimulai ---")

    # Baca data mentah
    print(f"Membaca data mentah dari: {RAW_DATA_FILE}")
    try:
        if RAW_DATA_FILE.endswith('.xlsx'):
            df_raw_input = pd.read_excel(RAW_DATA_FILE, sheet_name="Sheet1")
        elif RAW_DATA_FILE.endswith('.csv'):
            df_raw_input = pd.read_csv(RAW_DATA_FILE, low_memory=False)
        else:
            print("FATAL: Format file tidak didukung."); exit()
        print(f"Data mentah berhasil dimuat: {len(df_raw_input)} baris.")
    except FileNotFoundError: print(f"FATAL: File mentah tidak ditemukan."); exit()
    except Exception as e: print(f"FATAL: Error membaca data mentah: {e}"); exit()

    if df_raw_input.empty: print("FATAL: Data mentah kosong."); exit()
    
    # Lakukan agregasi
    df_aggregated_output = aggregate_data_by_time_window(
        df_raw_input,
        TIME_WINDOW_DURATION,
        COLUMNS_TO_AGGREGATE,
        GROUPBY_COLS_BEFORE_TIMEWINDOW
    )

    # Simpan hasil agregasi ke Excel
    if df_aggregated_output is not None and not df_aggregated_output.empty:
        print(f"Menyimpan data yang sudah diagregasi ke: {AGGREGATED_OUTPUT_FILE}")
        try:
            # Buat direktori jika belum ada
            os.makedirs(os.path.dirname(AGGREGATED_OUTPUT_FILE) or '.', exist_ok=True)
            df_aggregated_output.to_excel(AGGREGATED_OUTPUT_FILE, index=False, engine='openpyxl')
            print("Data yang sudah diagregasi berhasil disimpan.")
        except Exception as e:
            print(f"FATAL: Gagal menyimpan data yang sudah diagregasi: {e}")
            print("Pastikan Anda memiliki library 'openpyxl' terinstal: pip install openpyxl")
    else:
        print("Agregasi data gagal atau menghasilkan DataFrame kosong, tidak ada yang disimpan.")
        
    print("--- Skrip Agregasi Data Selesai ---")

--- Skrip Agregasi Data per Jendela Waktu Dimulai ---
Membaca data mentah dari: D:/Kuliah-temp/Teknofest/program/dataset/encoded/encoded_5G_UL_fill.xlsx
Data mentah berhasil dimuat: 72480 baris.
Memulai agregasi data dengan jendela waktu: 1s...
  Kolom yang akan diagregasi: ['Longitude', 'Latitude', 'NR_UE_PCI_0', 'NR_UE_RSRP_0', 'NR_UE_RSRQ_0', 'NR_UE_SINR_0', 'NR_UE_Timing_Advance', 'NR_UE_Pathloss_DL_0', 'NR_UE_CCE_AggregationLev_0', 'NR_UE_Modulation_Avg_DL_0', 'NR_UE_RI_DL_0', 'NR_UE_Nbr_PCI_0', 'NR_UE_Nbr_RSRP_0', 'NR_UE_Nbr_RSRQ_0', 'NR_UE_Nbr_PCI_1', 'NR_UE_Nbr_RSRP_1', 'NR_UE_Nbr_RSRQ_1', 'NR_UE_Nbr_PCI_2', 'NR_UE_Nbr_RSRP_2', 'NR_UE_Nbr_RSRQ_2', 'NR_UE_Nbr_PCI_3', 'NR_UE_Nbr_RSRP_3', 'NR_UE_Nbr_RSRQ_3', 'NR_UE_Nbr_PCI_4', 'NR_UE_Nbr_RSRP_4', 'NR_UE_Nbr_RSRQ_4', 'NR_UE_Throughput_PDCP_DL', 'App_Throughput_DL', 'NR_UE_NACK_Rate_DL_0', 'NR_UE_Ack_As_Nack_DL_0', 'NR_UE_BLER_DL_0', 'NR_UE_Power_Tx_PUSCH_0', 'NR_UE_Power_Tx_PRACH_0', 'NR_UE_NACK_Rate_UL_0', 'NR_UE_RACH_Attempt', 'N